In [0]:
dbutils.widgets.text("catalogo", "mlops", "Catálogo")
dbutils.widgets.text("esquema", "frecuencia", "Esquema")
dbutils.widgets.text("experimento", "/Users/santiagopadu@gmail.com/frecuencia_rc", "Experimento")
dbutils.widgets.text("alphas", "0.0001,0.001,0.01,0.1", "Alphas")
dbutils.widgets.text("modelo", "frecuencia_rc", "Modelo")
dbutils.widgets.text("xgb_profundidades", "3,5", "XGB profundidades")
dbutils.widgets.text("xgb_tasas", "0.05,0.1", "XGB tasas")
dbutils.widgets.text("xgb_arboles", "300", "XGB árboles")


catalogo = dbutils.widgets.get("catalogo")
esquema = dbutils.widgets.get("esquema")
experimento = dbutils.widgets.get("experimento")
modelo_nombre = dbutils.widgets.get("modelo")
alphas = [float(a) for a in dbutils.widgets.get("alphas").split(",")]
xgb_profundidades = [int(x) for x in dbutils.widgets.get("xgb_profundidades").split(",")]
xgb_tasas = [float(x) for x in dbutils.widgets.get("xgb_tasas").split(",")]
xgb_arboles = int(dbutils.widgets.get("xgb_arboles"))

In [0]:
tabla_train = f"{catalogo}.{esquema}.polizas_train"
tabla_test = f"{catalogo}.{esquema}.polizas_test"

def version_delta(tabla):
    return spark.sql(f"DESCRIBE HISTORY {tabla} LIMIT 1").collect()[0]["version"]

version_train = version_delta(tabla_train)
version_test = version_delta(tabla_test)

train = spark.sql(f"SELECT * FROM {tabla_train} VERSION AS OF {version_train}").toPandas()
test = spark.sql(f"SELECT * FROM {tabla_test} VERSION AS OF {version_test}").toPandas()

print(len(train), version_train, len(test), version_test)

In [0]:
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import OneHotEncoder, KBinsDiscretizer, FunctionTransformer, StandardScaler
from sklearn.linear_model import PoissonRegressor
from xgboost import XGBRegressor

categoricas = ["Area", "VehBrand", "VehGas", "Region", "VehPower"]
en_tramos = ["VehAge", "DrivAge"]
directas = ["BonusMalus"]
logaritmicas = ["Density"]
variables = categoricas + en_tramos + directas + logaritmicas

categoricas_xgb = ["Area", "VehBrand", "VehGas", "Region"]
numericas_xgb = ["VehPower", "VehAge", "DrivAge", "BonusMalus", "Density"]

def crear_glm(alpha):
    preproceso = ColumnTransformer([
        ("categoricas", OneHotEncoder(handle_unknown="ignore"), categoricas),
        ("tramos", KBinsDiscretizer(n_bins=10, encode="onehot", strategy="quantile", subsample=None, quantile_method="averaged_inverted_cdf"), en_tramos),
        ("directas", "passthrough", directas),
        ("logaritmo", make_pipeline(FunctionTransformer(np.log), StandardScaler()), logaritmicas),
    ])
    return Pipeline([
        ("preproceso", preproceso),
        ("modelo", PoissonRegressor(alpha=alpha, solver="newton-cholesky", max_iter=300)),
    ])

def crear_xgb(max_depth, learning_rate, n_estimators):
    preproceso = ColumnTransformer([
        ("categoricas", OneHotEncoder(handle_unknown="ignore"), categoricas_xgb),
        ("numericas", "passthrough", numericas_xgb),
    ], sparse_threshold=0)
    return Pipeline([
        ("preproceso", preproceso),
        ("modelo", XGBRegressor(
            objective="count:poisson",
            max_depth=max_depth,
            learning_rate=learning_rate,
            n_estimators=n_estimators,
            tree_method="hist",
            n_jobs=-1,
            random_state=42,
        )),
    ])

def crear_modelo(familia, p):
    return crear_glm(**p) if familia == "glm" else crear_xgb(**p)

candidatos = [("glm", {"alpha": a}) for a in alphas] + [
    ("xgb", {"max_depth": d, "learning_rate": t, "n_estimators": xgb_arboles})
    for d in xgb_profundidades for t in xgb_tasas
]

X_train, X_test = train[variables], test[variables]
y_train = train["ClaimNb"] / train["Exposure"]
y_test = test["ClaimNb"] / test["Exposure"]
w_train, w_test = train["Exposure"], test["Exposure"]

In [0]:
import mlflow
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_poisson_deviance
from mlflow.models import infer_signature

mlflow.sklearn.autolog(disable=True)
mlflow.set_experiment(experimento)

datos_train = mlflow.data.load_delta(table_name=tabla_train, version=version_train)
datos_test = mlflow.data.load_delta(table_name=tabla_test, version=version_test)

def grafico_deciles(mu, df):
    g = df.assign(esperados=mu * df["Exposure"], decil=pd.qcut(mu, 10, labels=False, duplicates="drop") + 1)
    r = g.groupby("decil")[["ClaimNb", "esperados", "Exposure"]].sum()
    fig, ax = plt.subplots()
    ax.plot(r.index, r["ClaimNb"] / r["Exposure"], marker="o", label="Observada")
    ax.plot(r.index, r["esperados"] / r["Exposure"], marker="o", label="Predicha")
    ax.set_xlabel("Decil de frecuencia predicha")
    ax.set_ylabel("Siniestros por año")
    ax.legend()
    return fig

resultados = []

for familia, p in candidatos:
    nombre_run = familia + "_" + "_".join(f"{k}_{v}" for k, v in p.items())
    with mlflow.start_run(run_name=nombre_run):
        modelo = crear_modelo(familia, p).fit(X_train, y_train, modelo__sample_weight=w_train)
        mu_train = modelo.predict(X_train)
        mu_test = modelo.predict(X_test)

        mlflow.log_params({
            "familia": familia,
            **p,
            "tabla_train": tabla_train,
            "version_train": version_train,
            "tabla_test": tabla_test,
            "version_test": version_test,
        })
        mlflow.log_input(datos_train, context="training")
        mlflow.log_input(datos_test, context="test")

        m = {
            "devianza_train": mean_poisson_deviance(y_train, mu_train, sample_weight=w_train),
            "devianza_test": mean_poisson_deviance(y_test, mu_test, sample_weight=w_test),
            "ae_train": train["ClaimNb"].sum() / (mu_train * w_train).sum(),
            "ae_test": test["ClaimNb"].sum() / (mu_test * w_test).sum(),
        }
        mlflow.log_metrics(m)

        fig = grafico_deciles(mu_test, test)
        mlflow.log_figure(fig, "deciles_test.png")
        plt.close(fig)

        info = mlflow.sklearn.log_model(
            modelo,
            name="modelo",
            signature=infer_signature(X_train, mu_train),
            input_example=X_train.head(5),
        )
        resultados.append({"familia": familia, "params": p, "devianza_test": m["devianza_test"], "model_uri": info.model_uri})

In [0]:
from mlflow import MlflowClient
from mlflow.exceptions import MlflowException

nombre_modelo = f"{catalogo}.{esquema}.{modelo_nombre}"
decimales_empate = 3
cliente = MlflowClient()

def criterio(r):
    return (
        round(r["devianza_test"], decimales_empate),
        0 if r["familia"] == "glm" else 1,
        -r["params"].get("alpha", 0),
    )

mejor = sorted(resultados, key=criterio)[0]

version = mlflow.register_model(mejor["model_uri"], nombre_modelo)

cliente.update_model_version(
    name=nombre_modelo,
    version=version.version,
    description=f"{mejor['familia'].upper()} {mejor['params']}, devianza_test={mejor['devianza_test']:.5f}, entrenado con {tabla_train} v{version_train}",
)
etiquetas = {"familia": mejor["familia"], **mejor["params"], "devianza_test": round(mejor["devianza_test"], 5), "version_train": version_train}
for clave, valor in etiquetas.items():
    cliente.set_model_version_tag(nombre_modelo, version.version, clave, str(valor))

try:
    cliente.get_model_version_by_alias(nombre_modelo, "champion")
    alias = "challenger"
except MlflowException:
    alias = "champion"

cliente.set_registered_model_alias(nombre_modelo, alias, version.version)
print(nombre_modelo, "versión", version.version, "→", alias)